# 固定股数五会话标签：分钟自身回撤增量信息门

事前冻结同一Qlib父本、单资产原价整数股订单及五会话HOLD/SOXL/SOXX费后标签。控制为原39项、f696/账户状态和普通完成分钟价量；候选仅加`decision_own_drawdown-opening_drawdown`。四个固定Ridge拟合只用2021首次决策之前已成熟的2018–2020标签，2021/2022按固定年度/区块/动作门一次检查；失败不反号或调窗。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
warnings.filterwarnings('ignore',message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.core.recorder_run import publish_evidence_manifest
from src.analysis.stage1_fixed_share_drawdown_information import prepare_feature_panel,source_variation_gate,fit_information_screen
root=Path.cwd();card=root/'research/stage1_v5_fixed_share_drawdown_information_20261030';plan_bytes=(card/'plan.json').read_bytes();plan=json.loads(plan_bytes);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['provider_uri']/'qlib_store_manifest.json')==plan['provider_manifest_sha256']
for role in ('label_source','parent','frame'):
 s=plan[role];source=root/s['card']
 assert sha(source/'summary.json')==s['summary_sha256'] and sha(source/'verification_manifest.json')==s['manifest_sha256']
 assert json.loads((source/'nas_binding.json').read_text())['evidence']['sha256']==s['nas_archive_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
def read(role,name):
 s=plan[role];v,_=load_verified_recorder_objects(root/s['card'],expected_recorder_id=s['recorder_id'],artifacts=[name],expected_manifest_sha256=s['manifest_sha256']);return v[name]
labels=read('label_source','analysis/labels');book=read('parent','analysis/native/dated_f696_0');frame=read('frame','inputs/covered_frame').reset_index(drop=True)
assert len(labels)==385932 and len(frame)==1056 and book['audit']['status']=='passed'
print('verified Qlib Recorder and NAS source identities')

[37836:MainThread](2026-10-30 15:27:24,310) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[37836:MainThread](2026-10-30 15:27:24,312) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[37836:MainThread](2026-10-30 15:27:24,312) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


verified Qlib Recorder and NAS source identities


In [2]:
bars=D.features(['SOXL','SOXX'],['$close','$raw_volume','$price_valid'],start_time='2018-10-15 09:31:00',end_time='2022-12-22 16:00:00',freq='1min')
panel,source=prepare_feature_panel(labels,frame,book,bars,plan)
assert source['rows']==len(labels)==385932 and panel.decision_at.eq(panel.bar_end_at+pd.Timedelta(minutes=1)).all()
variation=source_variation_gate(panel,plan)
(card/'source_gate.json').write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'source':source,'variation':variation,'value_columns_accessed':False},ensure_ascii=False,indent=2)+'\n')
if not variation['passed']:raise ValueError('blocked: fixed intraday own-drawdown source variation gate failed')
print(json.dumps({'source':source,'variation':variation},ensure_ascii=False,indent=2))

{
  "source": {
    "rows": 385932,
    "maximum_decision_nav_error_usd": 5.820766091346741e-11,
    "minute_features_available_rows": 385932
  },
  "variation": {
    "training_days_with_gt_0_1pp_intraday_range": 558,
    "maximum_abs_within_day_ordinary_rank_correlation": 0.213421355293225,
    "correlations": {
      "minute_soxl_ret5": -0.1279520847797985,
      "minute_soxl_ret15": -0.21178942574714513,
      "minute_soxx_ret5": -0.1302722510465568,
      "minute_soxx_ret15": -0.213421355293225,
      "minute_soxl_log1p_raw_volume": 0.028311078885515536,
      "minute_soxx_log1p_raw_volume": 0.032067731609966364
    },
    "passed": true
  }
}


In [3]:
screen=fit_information_screen(panel,labels,plan)
metrics=[{'model':model,'target':target,'year':int(year),**value} for (model,target,year),value in screen['metrics'].items()]
metric_map={(x['model'],x['target'],x['year']):x for x in metrics}
annual=screen['annual_actions'];byyear={x['year']:x for x in annual}
gates={}
gates['control_beats_training_mean']=all(metric_map[('control',target,y)]['day_mse']<metric_map[('control',target,y)]['train_mean_mse'] for target in plan['targets'] for y in plan['evaluation_years'])
gates['candidate_beats_control_mse']=all(metric_map[('candidate',target,y)]['day_mse']<metric_map[('control',target,y)]['day_mse'] for target in plan['targets'] for y in plan['evaluation_years'])
gates['relative_feature_sign_negative']=screen['relative_feature_coefficient']<0
gates['candidate_control_funded_action_change']=sum(x['candidate_control_action_changed_days'] for x in annual)>=plan['gates']['candidate_vs_control_funded_action_changed_days_min']
gates['same_model_zero_action_change']=sum(x['same_model_zero_action_changed_days'] for x in annual)>=plan['gates']['direct_zero_ablation_action_changed_days_min']
gates['candidate_control_value_positive_each_year']=all(x['candidate_minus_control_value_sum']>0 for x in annual)
gates['candidate_hold_value_positive_each_year']=all(x['candidate_value_sum']>0 for x in annual)
gates['candidate_control_block_lower_positive']=screen['block_lower']>0
gates['candidate_tail_not_worse_each_year']=all(x['candidate_minus_control_tail_damage_sum']<=0 for x in annual)
passed=all(gates.values())
daily=screen['daily_actions'].copy();daily['quarter']=daily.trade_date.dt.to_period('Q').astype(str)
quarter=[]
for q,part in daily.groupby('quarter',sort=True):
 quarter.append({'quarter':q,'days':len(part),'candidate_value_sum':float(part.value_candidate.sum()),'control_value_sum':float(part.value_control.sum()),'candidate_minus_control_value_sum':float(part.candidate_minus_control.sum()),'candidate_minus_control_tail_sum':float(part.tail_candidate_minus_control.sum())})
pred=pd.DataFrame({'trade_date':pd.to_datetime(screen['evaluation_day_index']),'control_soxl':screen['predictions'][('control','soxl_value')],'control_soxx':screen['predictions'][('control','soxx_value')],'candidate_soxl':screen['predictions'][('candidate','soxl_value')],'candidate_soxx':screen['predictions'][('candidate','soxx_value')],'candidate_zero_soxl':screen['zero_predictions']['soxl_value'],'candidate_zero_soxx':screen['zero_predictions']['soxx_value']})
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'screen_code_sha256':sha(root/'src/analysis/stage1_fixed_share_drawdown_information.py'),'source_variation':variation,'train_rows':screen['train_rows'],'train_days':screen['train_days'],'evaluation_rows':screen['eval_rows'],'evaluation_days':screen['eval_days'],'metrics':metrics,'relative_feature_coefficient':screen['relative_feature_coefficient'],'prediction_changed_rows':screen['prediction_changed_rows'],'same_model_zero_score_changed_rows':screen['same_model_zero_score_changed_rows'],'annual_actions':annual,'quarter_actions':quarter,'candidate_control_block_lower':screen['block_lower'],'gates':gates,'information_gate_pass':passed,'new_model_fits':4,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'fixed_minute_own_drawdown_information_pass_needs_continuous_target' if passed else 'not_promoted_fixed_minute_own_drawdown_information_failed'}
print(json.dumps(audit,ensure_ascii=False,indent=2))

[37836:MainThread](2026-10-30 15:27:27,085) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[37836:MainThread](2026-10-30 15:27:27,085) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[37836:MainThread](2026-10-30 15:27:27,086) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


{
  "experiment": "stage1_v5_fixed_share_drawdown_information_20261030",
  "plan_sha256": "5b04703ad61dac8799a0e5fbb83a57649ce5fafab5bb7617801aaba08e63630a",
  "screen_code_sha256": "66a7847ff3749b06126e23e3b87e421b846214ec40b6d95a72a724a62c55c903",
  "source_variation": {
    "training_days_with_gt_0_1pp_intraday_range": 558,
    "maximum_abs_within_day_ordinary_rank_correlation": 0.213421355293225,
    "correlations": {
      "minute_soxl_ret5": -0.1279520847797985,
      "minute_soxl_ret15": -0.21178942574714513,
      "minute_soxx_ret5": -0.1302722510465568,
      "minute_soxx_ret15": -0.213421355293225,
      "minute_soxl_log1p_raw_volume": 0.028311078885515536,
      "minute_soxx_log1p_raw_volume": 0.032067731609966364
    },
    "passed": true
  },
  "train_rows": 200403,
  "train_days": 549,
  "evaluation_rows": 180938,
  "evaluation_days": 494,
  "metrics": [
    {
      "model": "control",
      "target": "soxl_value",
      "year": 2021,
      "days": 252,
      "day_mse": 2

In [4]:
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,uri=str(tracking)):
 rec=R.get_recorder()
 rec.save_objects(artifact_path='inputs',plan=plan,source_gate=json.loads((card/'source_gate.json').read_text()))
 rec.save_objects(artifact_path='sources',screen_code=(root/'src/analysis/stage1_fixed_share_drawdown_information.py').read_text())
 rec.save_objects(artifact_path='models',control_soxl=screen['models'][('control','soxl_value')],control_soxx=screen['models'][('control','soxx_value')],candidate_soxl=screen['models'][('candidate','soxl_value')],candidate_soxx=screen['models'][('candidate','soxx_value')])
 rec.save_objects(artifact_path='analysis',predictions=pred,daily_actions=daily,annual_actions=annual,quarter_actions=quarter,metrics=metrics,audit=audit)
 rec.log_metrics(train_days=float(screen['train_days']),evaluation_days=float(screen['eval_days']),action_changed_days=float(sum(x['candidate_control_action_changed_days'] for x in annual)),candidate_control_block_lower=float(screen['block_lower']),information_gate_pass=float(passed))
assert rec.status=='FINISHED'
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'decision':audit['decision'],'historical_research_passed':False,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'train_rows':screen['train_rows'],'train_days':screen['train_days'],'evaluation_rows':screen['eval_rows'],'evaluation_days':screen['eval_days'],'source_variation':variation,'gates':gates,'annual_actions':annual,'quarter_actions':quarter,'candidate_control_block_lower':screen['block_lower'],'new_model_fits':4,'new_candidate_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*','models/*','analysis/*']},'limitations':['Fixed 5-session local action-value screen with overlapping paths, not a full candidate account or independent forward validation.','2021/22 history was the single frozen information evaluation; no retuning after seeing it.','Historical minute first receipt, actual broker execution and adjusted-price dividend cash remain unqualified.']}
(card/'workflow.yaml').write_text('experiment: '+card.name+'\nnotebook: '+card.name+'.ipynb\nrecorder_uri: '+str(tracking)+'\nartifact_policy: slim_card_recorder_first\n')
publish_evidence_manifest(card,summary)
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'recorder_id':rec.id,'information_gate_pass':passed,'train_days':screen['train_days'],'eval_days':screen['eval_days']},ensure_ascii=False))

[37836:MainThread](2026-10-30 15:27:29,775) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_v5_fixed_share_drawdown_information_20261030.


[37836:MainThread](2026-10-30 15:27:29,777) INFO - qlib.workflow - [exp.py:258] - Experiment 690974534629752934 starts running ...


[37836:MainThread](2026-10-30 15:27:29,890) INFO - qlib.workflow - [recorder.py:345] - Recorder 3f3ee86e5372470faa6825f63ae25bbf starts running under Experiment 690974534629752934 ...


[37836:MainThread](2026-10-30 15:27:30,583) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"recorder_id": "3f3ee86e5372470faa6825f63ae25bbf", "information_gate_pass": false, "train_days": 549, "eval_days": 494}


## 中文结论

SOXX历史Goal未通过，固定分钟自身回撤信息版本关闭。2018–2020成熟200403行/549日训练4个Qlib DatasetH/Ridge头，2021/2022有效180938行/494日一次评估；来源字段558日有日内变化且相对普通分钟字段最大秩相关0.213，候选/控制分数全行变化、同模型归零改变180575行，方向系数−2.07e−5。但实际候选/控制首行动只差15/3日，2022Q2两者62日全在09:48买SOXX、局部价值和−0.12933；2021Q3/Q4候选有少量增益，2021Q1损伤。控制/候选留出年MSE均差于训练均值常数，2022候选对HOLD值和−0.10421，区块下界−2.42e−5且尾部比控制差。全部冻结门未过，不调参/改窗/挑季度、不跑候选账户。Qlib Recorder `3f3ee86e5372470faa6825f63ae25bbf`、NAS档案及七件对象限定读回通过；完整原因和8季度见conclusion.md与failure_attribution.json。